# EEG Data Plotting

In [ ]:
# SETUP
from google.colab import drive
drive.mount('/content/drive')

# Install necessary packages (run once at the start of your notebook/script)
!pip install mne
import mne
import numpy as np
import matplotlib.pyplot as plt

# === Step 1: Load raw EEG data ===
file_path = '/content/drive/MyDrive/2025_Summer (Distal Speech Rate)/Project_Code/EEGData/12/12_annotated_raw.fif'
raw = mne.io.read_raw_fif(file_path, preload=True)

# === Step 2: Basic info ===
sfreq = raw.info['sfreq']
n_times = raw.n_times
duration_sec = raw.times[-1]

print(f"Sampling rate: {sfreq} Hz")
print(f"Total number of samples: {n_times}")
print(f"Total recording duration: {duration_sec / 60:.2f} minutes")

# === Step 3: Extract all events ===
events, event_id = mne.events_from_annotations(raw)
print("\nAll available event types:")
for key, val in event_id.items():
    print(f"  {key}: {val}")

# === Step 4: Filter for word onset events (e.g., 'Wd2E', 'Wd2N') ===
word_event_keys = [k for k in event_id.keys() if "Wd" in k]
print(f"\nFiltered word onset event types: {word_event_keys}")

# Combine word onset events from all relevant types
word_events = [events[events[:, 2] == event_id[k]] for k in word_event_keys]
word_events_all = np.vstack(word_events)
word_onset_times_sec = word_events_all[:, 0] / sfreq

print(f"\nTotal number of word onsets: {len(word_onset_times_sec)}")
print("First 5 word onset times (in seconds):", word_onset_times_sec[:5])

# === Step 5: Plot histogram of word onset times ===
plt.figure(figsize=(10, 4))
plt.hist(word_onset_times_sec, bins=50, color='skyblue', edgecolor='k')
plt.xlabel("Time (seconds)")
plt.ylabel("Word onset count")
plt.title("Distribution of Word Onsets in EEG Recording")
plt.tight_layout()
plt.show()


# Simulated PAC

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import hilbert

# Updated simulation: stronger PAC, wider kernel
sfreq = 250
duration = 3000  # shorter for testing
n_samples = int(sfreq * duration)
times = np.arange(n_samples) / sfreq

# --- Low-freq theta phase ---
lf_freq = 4
lf_signal = np.sin(2 * np.pi * lf_freq * times)
lf_phase = np.angle(hilbert(lf_signal))

# --- Word onsets ---
n_words = 1000
word_locs = np.random.randint(int(5 * sfreq), n_samples - int(5 * sfreq), size=n_words)
word_locs.sort()
word_locs_normal = word_locs[:n_words // 2]
word_locs_slowed = word_locs[n_words // 2:]

# --- Stronger PAC modulator with sustained width ---
# use all the context
amp_mod = np.zeros(n_samples)
amp_mod[word_locs_normal] = 1.0
amp_mod[word_locs_slowed] = 0.0  # minimal PAC in slowed condition

kernel_width = int(0.3 * sfreq)  # 300ms kernel
kernel = np.exp(-np.linspace(-2, 2, 2 * kernel_width) ** 2 / 0.1)
kernel /= kernel.max()
amp_mod = np.convolve(amp_mod, kernel, mode='same')

# --- Independent gamma amplitude ---
hf_amp_ind = np.zeros(n_samples)
for _ in range(5):
    f_i = np.random.uniform(0.05, 0.5)
    phi_i = np.random.uniform(-2 * np.pi, 2 * np.pi)
    hf_amp_ind += np.cos(2 * np.pi * f_i * times + phi_i)
hf_amp_ind = (hf_amp_ind - hf_amp_ind.min()) / (hf_amp_ind.max() - hf_amp_ind.min())

# --- Final modulated HF envelope ---
sigma = np.pi / 2
f_phi_l = np.exp(-(lf_phase ** 2) / (2 * sigma ** 2))
hf_envelope = (1 - amp_mod) * hf_amp_ind + amp_mod * f_phi_l

# --- Gamma-band carrier with PAC ---
hf_freq = 80
hf_carrier = np.sin(2 * np.pi * hf_freq * times)
hf_signal = hf_envelope * hf_carrier + 0.3 * np.random.randn(n_samples)

# --- Plot zoomed in on a normal-context word onset ---
onset = word_locs_normal[0]
win = int(1 * sfreq)
start, end = onset - win, onset + win
plt.figure(figsize=(10, 4))
plt.plot(times[start:end], hf_signal[start:end], color='lightgrey', label='Simulated EEG')
plt.plot(times[start:end], hf_envelope[start:end], label='PAC Envelope')
plt.plot(times[start:end], amp_mod[start:end], label='amp_mod(t)', color='darkorange')
plt.plot(times[start:end], f_phi_l[start:end], label='f(ϕ_L(t))', linestyle=':', color='green')
plt.xlabel("Time (s)")
plt.ylabel("Amplitude")
plt.title("PAC Simulation (Ideal Modulation at Word Onset)")
plt.legend()
plt.tight_layout()
plt.show()


# Test mTRF on hf_envelope

X1 = f_phi_l: phase-based predictor (1D)

X2 = amp_mod: strength modulator (1D)

X3 = interaction = f_phi_l * amp_mod: interaction term

Y = hf_envelope: the ground-truth PAC envelope

In [ ]:
from sklearn.linear_model import Ridge
from scipy.linalg import toeplitz

def make_lagged_matrix(x, lags):
    """Construct lagged predictor matrix with given lags."""
    n = len(x)
    lagged = np.stack([np.roll(x, -lag) for lag in lags], axis=1)
    lagged[:lags[-1], :] = 0  # zero-padding
    return lagged

sfreq = 250  # 1 kHz
tmin, tmax = -0.3, 0.6  # 600ms in total
lags = np.arange(int(tmin * sfreq), int(tmax * sfreq))  # e.g., -300 to +299 ms

X_lagged = np.hstack([
    make_lagged_matrix(f_phi_l, lags),
    make_lagged_matrix(amp_mod, lags),
    make_lagged_matrix(f_phi_l * amp_mod, lags)
])

ridge = Ridge(alpha=1.0)  # regularization strength
ridge.fit(X_lagged, hf_envelope)
w = ridge.coef_  # shape: (n_lags * n_predictors,)

trf_f_phi = w[:len(lags)]
trf_amp_mod = w[len(lags):2*len(lags)]
trf_interact = w[2*len(lags):]

time_lags_ms = lags * 1000 / sfreq

plt.figure(figsize=(10, 4))
plt.plot(time_lags_ms, trf_f_phi, label='TRF: f(ϕ_L(t))')
plt.plot(time_lags_ms, trf_amp_mod, label='TRF: amp_mod(t)')
plt.plot(time_lags_ms, trf_interact, label='TRF: interaction')
plt.axhline(0, color='black', linewidth=0.5)
plt.xlabel("Lag (ms)")
plt.ylabel("TRF Weight")
plt.title("mTRF Weights for PAC Simulation")
plt.legend()
plt.tight_layout()
plt.show()


how each predictor (f(ϕ_L), amp_mod(t), and their interaction) explains the high-frequency EEG envelope (hf_envelope) over time lags around each event.
1.  TRF: f(ϕ_L(t)) (blue)
Flat and low in amplitude.

This is expected: the low-frequency phase shape f_phi_l varies smoothly across time and is always present, so it doesn't align tightly to word onsets. Therefore, it has low predictive value for hf_envelope on its own.

2. TRF: amp_mod(t) (orange)
Also relatively flat, slight shape across time.

This makes sense too: amp_mod(t) is sparse, aligned to word onsets, but its direct influence on hf_envelope is modulated via interaction with phase (i.e., not additive on its own). So again, only modest contribution to prediction.

3. TRF: interaction (f_phi_l × amp_mod) (green)
Strong, sharp positive peak around 0 ms lag.

a PAC effect where the low-frequency phase modulates high-frequency amplitude, but only around word onsets.

Since the interaction term is only nonzero at word onsets and carries structured phase-shape, it cleanly predicts peaks in the hf_envelope. At the exact moment of word onset (lag = 0 ms), the brain shows enhanced coupling between low-frequency phase and high-frequency amplitude.


# permutation testing
We want to know:

Are the mTRF weights learned from f(ϕ_L(t)) × amp_mod(t) significantly better than chance?

We'll test this by:

Shuffling amp_mod(t) (or f_phi_l) to break the PAC structure

Refitting mTRFs with shuffled predictors

Building a null distribution of TRF weights

Comparing your real TRF weights against this distribution



In [ ]:
import numpy as np
from sklearn.linear_model import Ridge
import matplotlib.pyplot as plt
from scipy.stats import percentileofscore

# Simulated variables
sfreq = 250
duration = 60
n_samples = sfreq * duration
times = np.arange(n_samples) / sfreq

# Simulate data matching the user's PAC model
lf_freq = 4
lf_signal = np.sin(2 * np.pi * lf_freq * times)
lf_phase = np.angle(np.fft.ifft(np.fft.fft(lf_signal)))
f_phi_l = np.exp(-(lf_phase ** 2) / (2 * (np.pi / 2) ** 2))

word_onsets = np.zeros(n_samples)
word_locs = np.linspace(2 * sfreq, n_samples - 2 * sfreq, 20).astype(int)
word_onsets[word_locs] = 1

amp_mod = np.convolve(word_onsets, np.exp(-np.linspace(-1, 1, 100)**2 / 0.05), mode='same')
amp_mod = 0.25 + 0.75 * (amp_mod - amp_mod.min()) / (amp_mod.max() - amp_mod.min())

hf_amp_ind = np.random.rand(n_samples)
hf_envelope = (1 - amp_mod) * hf_amp_ind + amp_mod * f_phi_l

# --- Lag matrix construction ---
def make_lagged_matrix(x, lags):
    return np.stack([np.roll(x, -lag) for lag in lags], axis=1)

lag_times = np.arange(-100, 401, 20)
lags = (lag_times / 1000 * sfreq).astype(int)

X_fphi = make_lagged_matrix(f_phi_l, lags)
X_amp = make_lagged_matrix(amp_mod, lags)
X_inter = make_lagged_matrix(f_phi_l * amp_mod, lags)

Y = hf_envelope

X_real = np.hstack([X_fphi, X_amp, X_inter])
ridge = Ridge(alpha=1.0)
ridge.fit(X_real, Y)
w_real = ridge.coef_

# --- Permutation test ---
n_permutations = 500
w_nulls = np.zeros((n_permutations, len(w_real)))

for i in range(n_permutations):
    amp_mod_shuffled = np.random.permutation(amp_mod)
    X_amp_shuff = make_lagged_matrix(amp_mod_shuffled, lags)
    X_inter_shuff = make_lagged_matrix(f_phi_l * amp_mod_shuffled, lags)
    X_null = np.hstack([X_fphi, X_amp_shuff, X_inter_shuff])

    model_null = Ridge(alpha=1.0)
    model_null.fit(X_null, Y)
    w_nulls[i, :] = model_null.coef_

# --- Compute p-values for interaction term ---
real_inter = w_real[2 * len(lags):]
null_inter = w_nulls[:, 2 * len(lags):]

p_vals = np.array([
    1 - (percentileofscore(null_inter[:, j], real_inter[j]) / 100)
    for j in range(len(lags))
])

# --- Plotting ---
plt.figure(figsize=(10, 4))
plt.plot(lag_times, real_inter, label='Real TRF: interaction', color='green')
plt.fill_between(
    lag_times, 0, real_inter,
    where=p_vals < 0.05, color='red', alpha=0.3, label='p < 0.05'
)
plt.axhline(0, color='black', linewidth=0.5)
plt.xlabel("Lag (ms)")
plt.ylabel("TRF Weight")
plt.title("Significant TRF Weights for PAC Interaction")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
from sklearn.linear_model import Ridge
import matplotlib.pyplot as plt
from scipy.stats import percentileofscore

# === Parameters ===
sfreq = 250
lags = np.arange(-100, 400+1, 10)  # in ms
lag_samples = (lags / 1000 * sfreq).astype(int)

def make_lagged_matrix(x, lags):
    return np.stack([np.roll(x, -lag) for lag in lags], axis=1)

# === Predictor: binary word onsets ===
X_onset = make_lagged_matrix(word_onsets, lag_samples)

# === Target: HF envelope ===
Y = exp(-sqrt(-1)*lf_phase)*hf_envelope

#-- recover the kernel-

# === Fit TRF Model ===
ridge = Ridge(alpha=1.0)
ridge.fit(X_onset, Y)
w_real = ridge.coef_

# === Permutation Test ===
n_permutations = 500
w_nulls = np.zeros((n_permutations, len(lag_samples)))

for i in range(n_permutations):
    shuffled_onsets = np.random.permutation(word_onsets)
    X_shuff = make_lagged_matrix(shuffled_onsets, lag_samples)

    ridge_null = Ridge(alpha=1.0)
    ridge_null.fit(X_shuff, Y)
    w_nulls[i, :] = ridge_null.coef_

# === Compute p-values ===
p_vals = np.array([
    1 - (percentileofscore(w_nulls[:, j], w_real[j]) / 100)
    for j in range(len(lag_samples))
])

# === Plot ===
plt.figure(figsize=(10, 4))
plt.plot(lags, abs(w_real, label='TRF: Word Onset', color='blue'))
plt.fill_between(
    lags, 0, np.abs(w_real),
    where=p_vals < 0.05, color='red', alpha=0.3, label='p < 0.05'
)
plt.axhline(0, color='black', linewidth=0.5)
plt.xlabel("Lag (ms)")
plt.ylabel("TRF Weight")
plt.title("TRF: High-Frequency Envelope Predictable from Word Onsets")
plt.legend()
plt.tight_layout()
plt.show()
